# Advanced Modeling — CatBoost + Optuna Tuning + Stacking Ensemble

Goal: meaningfully improve on notebook 05's best R² of **0.444** (Ensemble LGB+XGB).

| Technique | Expected Gain |
|---|---|
| 22 additional interaction/composite features (85 total) | +0.01–0.03 |
| CatBoost baseline | comparable / complementary |
| Optuna hyperparameter search (LGB + XGB, 60+50 trials) | +0.01–0.02 |
| Optimized 3-model ensemble (LGB + XGB + CatBoost) | +0.005–0.01 |
| Stacking with Ridge meta-learner | +0.005–0.01 |
| Raw MEPS enrichment (smoking, Rx fills, functional lims) | +0.02–0.05 if available |

All outputs consumed by the next section; test set is **never touched until final evaluation**.

In [21]:
# Section 0: Install dependencies
import subprocess, sys

for pkg in ['catboost', 'optuna', 'lightgbm']:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f'WARNING: {pkg} install may have failed:', result.stderr[:200])
    else:
        print(f'{pkg}: installed ok')

catboost: installed ok
optuna: installed ok
lightgbm: installed ok


In [22]:
import pandas as pd
import numpy as np
import joblib
import json
import os
import warnings
import copy
warnings.filterwarnings('ignore')

import lightgbm as lgb
from xgboost import XGBRegressor
import catboost as cb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge

print(f'LightGBM  : {lgb.__version__}')
import xgboost; print(f'XGBoost   : {xgboost.__version__}')
print(f'CatBoost  : {cb.__version__}')
print(f'Optuna    : {optuna.__version__}')

LightGBM  : 4.6.0
XGBoost   : 3.1.3
CatBoost  : 1.2.10
Optuna    : 4.7.0


## 1. Load Data

In [23]:
X_train = pd.read_csv('X_train.csv').reset_index(drop=True)
X_test  = pd.read_csv('X_test.csv').reset_index(drop=True)
y_train = pd.read_csv('y_train.csv').squeeze().reset_index(drop=True)
y_test  = pd.read_csv('y_test.csv').squeeze().reset_index(drop=True)

print(f'Train : {X_train.shape[0]:,} rows × {X_train.shape[1]} features')
print(f'Test  : {X_test.shape[0]:,} rows × {X_test.shape[1]} features')
print(f'Target (log1p scale): mean={y_train.mean():.3f}  std={y_train.std():.3f}')

zero_count = int((y_train == 0).sum())
zero_pct   = 100.0 * zero_count / len(y_train)
print(f'\nZero spenders: {zero_count:,} ({zero_pct:.1f}% of training rows)')

Train : 17,944 rows × 55 features
Test  : 4,487 rows × 55 features
Target (log1p scale): mean=6.563  std=3.215

Zero spenders: 2,722 (15.2% of training rows)


## 2. Feature Engineering

Two layers:
1. `add_derived_features()` — same 8 features as notebook 05 (63 total)
2. `add_advanced_features()` — 22 additional interaction / composite features → **85 total**

All are derived from the existing 55 base columns; no new user inputs required.

In [24]:
DX_COLS = [
    'dx_hypertension', 'dx_coronary_heart_disease', 'dx_angina',
    'dx_myocardial_infarction', 'dx_stroke', 'dx_emphysema',
    'dx_high_cholesterol', 'dx_cancer', 'dx_arthritis',
    'dx_asthma', 'dx_adhd_add', 'dx_diabetes',
]
CARDIAC_COLS = [
    'dx_coronary_heart_disease', 'dx_angina',
    'dx_myocardial_infarction', 'dx_stroke',
]


def add_derived_features(X: pd.DataFrame) -> pd.DataFrame:
    """Add 8 derived features (same as notebook 05)."""
    X = X.copy()
    X['num_conditions']   = X[DX_COLS].sum(axis=1).astype(float)
    X['is_elderly']       = (X['age'] >= 65).astype(float)
    X['bmi_obese']        = (X['bmi'] >= 30).astype(float)
    X['dual_eligible']    = ((X['has_medicare'] == 1) & (X['has_medicaid'] == 1)).astype(float)
    X['cardiac_burden']   = X[CARDIAC_COLS].sum(axis=1).astype(float)
    X['age_sq']           = (X['age'] ** 2) / 1000.0
    X['age_x_conditions'] = (X['age'] * X['num_conditions']) / 10.0
    X['health_burden']    = (X['self_rated_health'] * X['num_conditions']) / 10.0
    return X


def add_advanced_features(X: pd.DataFrame) -> pd.DataFrame:
    """Add 22 additional interaction / composite features (call AFTER add_derived_features)."""
    X = X.copy()

    # Insurance × condition risk
    X['uninsured_x_conditions'] = X['is_uninsured'] * X['num_conditions']
    X['medicaid_x_burden']      = X['has_medicaid'] * X['health_burden']
    X['medicare_x_cardiac']     = X['has_medicare'] * X['cardiac_burden']
    X['uninsured_x_cancer']     = X['is_uninsured'] * X['dx_cancer']
    X['uninsured_x_diabetes']   = X['is_uninsured'] * X['dx_diabetes']

    # Metabolic cluster
    X['metabolic_syndrome']      = (X['dx_diabetes'] * X['dx_hypertension']).astype(float)
    X['diabetic_obese']          = (X['dx_diabetes'] * X['bmi_obese']).astype(float)
    X['cardiometabolic_burden']  = (X['dx_diabetes'] + X['dx_hypertension'] + X['dx_high_cholesterol']).astype(float)

    # Mental health
    X['mental_x_conditions']  = (X['k6_distress_score'] / 24.0) * X['num_conditions']
    X['phq2_x_poor_health']   = X['phq2_depression_score'] * X['self_rated_health']

    # SES × health
    vulnerability = (6.0 - X['poverty_category'])
    X['poverty_x_poor_health'] = vulnerability * X['self_rated_health']
    X['poverty_x_conditions']  = vulnerability * X['num_conditions']

    # Elderly risk
    lim_total = X['needs_help_adl'] + X['needs_help_iadl'] + X['any_limitation']
    X['elderly_x_limitation']  = X['is_elderly'] * lim_total
    X['elderly_x_cardiac']     = X['is_elderly'] * X['cardiac_burden']
    X['elderly_x_cancer']      = X['is_elderly'] * X['dx_cancer']

    # Income per capita proxy (family_income is signed_log1p transformed)
    X['income_per_capita'] = X['family_income'] / (X['family_size'] + 0.1)

    # Polynomial
    X['k6_sq']        = (X['k6_distress_score'] / 24.0) ** 2
    X['bmi_deviation'] = ((X['bmi'] - 22.0) / 10.0) ** 2

    # Access × need mismatch
    X['chronic_no_usual_care'] = X['num_conditions'] * (1 - X['has_usual_care'])

    # Weighted disease severity (based on relative Medicare cost weights)
    X['weighted_severity'] = (
        2.0 * X['dx_cancer'] + 1.8 * X['dx_coronary_heart_disease'] +
        1.8 * X['dx_myocardial_infarction'] + 1.5 * X['dx_stroke'] +
        1.5 * X['dx_emphysema'] + 1.3 * X['dx_diabetes'] +
        1.2 * X['dx_hypertension'] + 1.1 * X['dx_high_cholesterol'] +
        1.0 * X['dx_arthritis'] + 0.9 * X['dx_asthma'] +
        0.7 * X['dx_angina'] + 0.5 * X['dx_adhd_add']
    )

    # Dual eligible × burden
    X['dual_x_burden'] = X['dual_eligible'] * X['health_burden']

    # Limitation severity composite
    X['limitation_severity'] = lim_total.astype(float)

    return X


# Apply both layers
X_train_adv = add_advanced_features(add_derived_features(X_train))
X_test_adv  = add_advanced_features(add_derived_features(X_test))

base_cols    = list(X_train.columns)           # 55
derived_cols = list(X_train_adv.columns[55:63])  # 8
advanced_cols = list(X_train_adv.columns[63:])   # 22

print(f'Base features    : {len(base_cols)}')
print(f'Derived features : {len(derived_cols)}  {derived_cols}')
print(f'Advanced features: {len(advanced_cols)}')
print(f'Total            : {X_train_adv.shape[1]}')
print()
print('Advanced feature names:')
for c in advanced_cols:
    print(f'  {c}')
print()
print('Advanced feature summary stats:')
print(X_train_adv[advanced_cols].describe().T[['mean', 'std', 'min', 'max']].round(3).to_string())

Base features    : 55
Derived features : 8  ['num_conditions', 'is_elderly', 'bmi_obese', 'dual_eligible', 'cardiac_burden', 'age_sq', 'age_x_conditions', 'health_burden']
Advanced features: 22
Total            : 85

Advanced feature names:
  uninsured_x_conditions
  medicaid_x_burden
  medicare_x_cardiac
  uninsured_x_cancer
  uninsured_x_diabetes
  metabolic_syndrome
  diabetic_obese
  cardiometabolic_burden
  mental_x_conditions
  phq2_x_poor_health
  poverty_x_poor_health
  poverty_x_conditions
  elderly_x_limitation
  elderly_x_cardiac
  elderly_x_cancer
  income_per_capita
  k6_sq
  bmi_deviation
  chronic_no_usual_care
  weighted_severity
  dual_x_burden
  limitation_severity

Advanced feature summary stats:
                         mean    std     min     max
uninsured_x_conditions  0.038  0.300   0.000   8.000
medicaid_x_burden       0.111  0.391   0.000   5.500
medicare_x_cardiac      0.105  0.438   0.000   4.000
uninsured_x_cancer      0.001  0.037   0.000   1.000
uninsured_

## 3. Evaluation Helper + Validation Split

In [25]:
def evaluate(name, y_true, y_pred_log):
    """Compute metrics on log1p and dollar scales. All values cast to Python float."""
    r2       = float(r2_score(y_true, y_pred_log))
    rmse_log = float(np.sqrt(mean_squared_error(y_true, y_pred_log)))
    mae_log  = float(mean_absolute_error(y_true, y_pred_log))
    y_true_d = np.expm1(np.array(y_true, dtype=float))
    y_pred_d = np.maximum(np.expm1(np.array(y_pred_log, dtype=float)), 0.0)
    rmse_d   = float(np.sqrt(mean_squared_error(y_true_d, y_pred_d)))
    mae_d    = float(mean_absolute_error(y_true_d, y_pred_d))
    pred_mean = float(y_pred_d.mean())
    return dict(
        model=name,
        r2=round(r2, 4),
        rmse_log=round(rmse_log, 4),
        mae_log=round(mae_log, 4),
        rmse_dollar=round(rmse_d, 2),
        mae_dollar=round(mae_d, 2),
        pred_mean=round(pred_mean, 2),
    )


results = []

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_adv, y_train, test_size=0.15, random_state=42
)

print(f'Training   : {X_tr.shape[0]:,} rows')
print(f'Validation : {X_val.shape[0]:,} rows  (early stopping / Optuna only)')
print(f'Test       : {X_test_adv.shape[0]:,} rows  (final evaluation only)')

actual_mean = float(np.expm1(y_test.values).mean())
print(f'\nTest set actual mean spending: ${actual_mean:,.0f}')

Training   : 15,252 rows
Validation : 2,692 rows  (early stopping / Optuna only)
Test       : 4,487 rows  (final evaluation only)

Test set actual mean spending: $7,690


## 4. Zero-Inflation Analysis

~15% of respondents report zero healthcare spending. Understanding how they differ from positive spenders motivates the two-stage hurdle model.

In [26]:
zero_mask = (y_train.values == 0)
pos_mask  = ~zero_mask

print(f'Zero spenders    : {zero_mask.sum():,} ({100*zero_mask.mean():.1f}%)')
print(f'Positive spenders: {pos_mask.sum():,} ({100*pos_mask.mean():.1f}%)')
print()

compare_cols = [
    'age', 'self_rated_health', 'num_conditions',
    'has_medicare', 'is_uninsured', 'has_usual_care',
]

X_arr = X_train_adv.values
col_idx = {c: i for i, c in enumerate(X_train_adv.columns)}

header = '{:<28}  {:>18}  {:>21}'.format('Feature', 'Zero-spender mean', 'Positive-spender mean')
print(header)
print('-' * 72)
for col in compare_cols:
    idx = col_idx[col]
    z_mean = X_arr[zero_mask, idx].mean()
    p_mean = X_arr[pos_mask, idx].mean()
    print('  {:<26}  {:>18.3f}  {:>21.3f}'.format(col, z_mean, p_mean))

Zero spenders    : 2,722 (15.2%)
Positive spenders: 15,222 (84.8%)

Feature                        Zero-spender mean  Positive-spender mean
------------------------------------------------------------------------
  age                                     33.249                 45.130
  self_rated_health                        2.069                  2.354
  num_conditions                           0.379                  1.525
  has_medicare                             0.066                  0.300
  is_uninsured                             0.261                  0.037
  has_usual_care                           0.346                  0.768


## 5. Improved Two-Stage Hurdle (log1p space)

Improvement over notebook 04's hurdle: everything stays in log1p space so the regressor doesn't have to learn the log transformation implicitly.

- **Stage 1**: LGBMClassifier predicts P(y > 0)
- **Stage 2**: LGBMRegressor trained only on positive spenders
- **Threshold**: tuned on validation set to maximise R²

In [27]:
from lightgbm import LGBMClassifier, LGBMRegressor

# ── Stage 1: Classifier ──────────────────────────────────────────────────────
y_tr_binary  = (y_tr.values > 0).astype(int)
y_val_binary = (y_val.values > 0).astype(int)

clf = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=63,
    min_child_samples=20,
    n_jobs=-1,
    random_state=42,
    verbose=-1,
)
clf.fit(
    X_tr.values, y_tr_binary,
    eval_set=[(X_val.values, y_val_binary)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=-1),
    ],
)
print(f'Classifier best iteration: {clf.best_iteration_}')

# ── Stage 2: Regressor on positive-spender rows only ────────────────────────
pos_tr_mask = (y_tr.values > 0)
X_tr_pos    = X_tr.values[pos_tr_mask]
y_tr_pos    = y_tr.values[pos_tr_mask]

# Sub-split positive rows for early stopping
X_tr_pos_sub, X_val_pos, y_tr_pos_sub, y_val_pos = train_test_split(
    X_tr_pos, y_tr_pos, test_size=0.15, random_state=42
)

reg = LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.02,
    num_leaves=127,
    min_child_samples=20,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    n_jobs=-1,
    random_state=42,
    verbose=-1,
)
reg.fit(
    X_tr_pos_sub, y_tr_pos_sub,
    eval_set=[(X_val_pos, y_val_pos)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=75, verbose=False),
        lgb.log_evaluation(period=-1),
    ],
)
print(f'Regressor best iteration : {reg.best_iteration_}')

# ── Threshold tuning on validation ──────────────────────────────────────────
val_prob    = clf.predict_proba(X_val.values)[:, 1]
val_reg_pred = reg.predict(X_val.values)

best_thresh = 0.5
best_val_r2 = -np.inf
for thresh in np.arange(0.1, 0.9, 0.05):
    y_val_hurdle = np.where(val_prob >= thresh, val_reg_pred, 0.0)
    val_r2 = r2_score(y_val.values, y_val_hurdle)
    if val_r2 > best_val_r2:
        best_val_r2 = val_r2
        best_thresh = thresh

print(f'Best threshold: {best_thresh:.2f}  (val R²={best_val_r2:.4f})')

# ── Evaluate on test set ─────────────────────────────────────────────────────
test_prob     = clf.predict_proba(X_test_adv.values)[:, 1]
test_reg_pred = reg.predict(X_test_adv.values)
y_pred_hurdle = np.where(test_prob >= best_thresh, test_reg_pred, 0.0)

res_hurdle = evaluate('Hurdle (LGB log1p space)', y_test.values, y_pred_hurdle)
results.append(res_hurdle)
print(f"Test R²: {res_hurdle['r2']:.4f}  MAE: ${res_hurdle['mae_dollar']:,.0f}")

Classifier best iteration: 225
Regressor best iteration : 159
Best threshold: 0.50  (val R²=0.2852)
Test R²: 0.2804  MAE: $6,210


## 6. CatBoost Baseline

CatBoost uses oblivious trees and symmetric splits that make it robust to overfitting and complementary to LightGBM / XGBoost in an ensemble.

In [28]:
print('Training CatBoost baseline...')

cb_model = cb.CatBoostRegressor(
    iterations=2000,
    learning_rate=0.03,
    depth=7,
    l2_leaf_reg=3.0,
    loss_function='RMSE',
    eval_metric='RMSE',
    early_stopping_rounds=75,
    random_seed=42,
    verbose=0,
)
cb_model.fit(
    X_tr.values, y_tr.values,
    eval_set=(X_val.values, y_val.values),
)

y_pred_cb = cb_model.predict(X_test_adv.values)
res_cb = evaluate('CatBoost (85 feat)', y_test.values, y_pred_cb)
results.append(res_cb)

print(f"Best iteration: {cb_model.get_best_iteration()}")
print(f"Test R²: {res_cb['r2']:.4f}  MAE: ${res_cb['mae_dollar']:,.0f}  Pred mean: ${res_cb['pred_mean']:,.0f}")

Training CatBoost baseline...
Best iteration: 1118
Test R²: 0.4460  MAE: $6,163  Pred mean: $2,921


## 7. Optuna Hyperparameter Optimization — LightGBM

Objective: maximise validation R². Test set is **not** used during search.

In [29]:
def lgb_objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 500, 4000),
        'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 63, 511),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'feature_fraction':  trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction':  trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq':      1,
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'n_jobs':            -1,
        'random_state':      42,
        'verbose':           -1,
    }
    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_tr.values, y_tr.values,
        eval_set=[(X_val.values, y_val.values)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=75, verbose=False),
            lgb.log_evaluation(period=-1),
        ],
    )
    return float(r2_score(y_val.values, model.predict(X_val.values)))


print('Running Optuna LightGBM search (60 trials)...')
lgb_study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
lgb_study.optimize(lgb_objective, n_trials=60, show_progress_bar=False)

lgb_best_params = {k: (int(v) if isinstance(v, (np.integer,)) else (float(v) if isinstance(v, (np.floating,)) else v))
                   for k, v in lgb_study.best_params.items()}
print(f'Best val R²  : {lgb_study.best_value:.4f}')
print('Best params  :', lgb_best_params)

# Train final model on full training data
print('\nTraining final LGB on full X_train_adv...')
lgb_opt = lgb.LGBMRegressor(**lgb_best_params, verbose=-1, n_jobs=-1, random_state=42)
lgb_opt.fit(
    X_train_adv.values, y_train.values,
    eval_set=[(X_val.values, y_val.values)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=75, verbose=False),
        lgb.log_evaluation(period=-1),
    ],
)

y_pred_lgb_opt = lgb_opt.predict(X_test_adv.values)
res_lgb_opt = evaluate('LGB Optuna (85 feat)', y_test.values, y_pred_lgb_opt)
results.append(res_lgb_opt)
print(f"Test R²: {res_lgb_opt['r2']:.4f}  MAE: ${res_lgb_opt['mae_dollar']:,.0f}  Pred mean: ${res_lgb_opt['pred_mean']:,.0f}")

Running Optuna LightGBM search (60 trials)...
Best val R²  : 0.4339
Best params  : {'n_estimators': 655, 'learning_rate': 0.009493833643558002, 'num_leaves': 65, 'min_child_samples': 42, 'feature_fraction': 0.5729441512430841, 'bagging_fraction': 0.6691267653976442, 'reg_alpha': 0.0017258780200462903, 'reg_lambda': 7.701602609506348}

Training final LGB on full X_train_adv...
Test R²: 0.4443  MAE: $6,163  Pred mean: $2,745


## 8. Optuna Hyperparameter Optimization — XGBoost

In [30]:
def xgb_objective(trial):
    params = {
        'n_estimators':       trial.suggest_int('n_estimators', 500, 4000),
        'learning_rate':      trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'max_depth':          trial.suggest_int('max_depth', 4, 9),
        'min_child_weight':   trial.suggest_int('min_child_weight', 1, 10),
        'subsample':          trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':   trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'colsample_bylevel':  trial.suggest_float('colsample_bylevel', 0.5, 1.0),
        'reg_alpha':          trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda':         trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'gamma':              trial.suggest_float('gamma', 0.0, 0.5),
        'n_jobs':             -1,
        'random_state':       42,
        'verbosity':          0,
        'early_stopping_rounds': 75,
    }
    model = XGBRegressor(**params)
    model.fit(
        X_tr.values, y_tr.values,
        eval_set=[(X_val.values, y_val.values)],
        verbose=False,
    )
    return float(r2_score(y_val.values, model.predict(X_val.values)))


print('Running Optuna XGBoost search (50 trials)...')
xgb_study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
xgb_study.optimize(xgb_objective, n_trials=50, show_progress_bar=False)

xgb_best_params = {k: (int(v) if isinstance(v, (np.integer,)) else (float(v) if isinstance(v, (np.floating,)) else v))
                   for k, v in xgb_study.best_params.items()}
print(f'Best val R²  : {xgb_study.best_value:.4f}')
print('Best params  :', xgb_best_params)

# Train final model — keep early_stopping_rounds for training with eval_set
print('\nTraining final XGB on full X_train_adv...')
xgb_opt = XGBRegressor(
    **xgb_best_params,
    n_jobs=-1,
    random_state=42,
    verbosity=0,
)
xgb_opt.fit(
    X_train_adv.values, y_train.values,
    eval_set=[(X_val.values, y_val.values)],
    verbose=False,
)

y_pred_xgb_opt = xgb_opt.predict(X_test_adv.values)
res_xgb_opt = evaluate('XGB Optuna (85 feat)', y_test.values, y_pred_xgb_opt)
results.append(res_xgb_opt)
print(f"Test R²: {res_xgb_opt['r2']:.4f}  MAE: ${res_xgb_opt['mae_dollar']:,.0f}  Pred mean: ${res_xgb_opt['pred_mean']:,.0f}")

Running Optuna XGBoost search (50 trials)...
Best val R²  : 0.4355
Best params  : {'n_estimators': 3916, 'learning_rate': 0.006673676627952104, 'max_depth': 4, 'min_child_weight': 10, 'subsample': 0.8599175287500006, 'colsample_bytree': 0.7207406937991798, 'colsample_bylevel': 0.5248195671968097, 'reg_alpha': 0.08939923926160181, 'reg_lambda': 0.25545967136260384, 'gamma': 0.030263133872909055}

Training final XGB on full X_train_adv...
Test R²: 0.4452  MAE: $6,140  Pred mean: $2,968


## 9. Optimized Ensemble (LGB + XGB + CatBoost)

Grid search over blend weights on the validation set. Three-model ensemble can reduce variance beyond the two-model version in notebook 05.

In [31]:
# Validation predictions from each model
val_lgb = lgb_opt.predict(X_val.values)
val_xgb = xgb_opt.predict(X_val.values)
val_cb  = cb_model.predict(X_val.values)

best_val_r2 = -np.inf
best_w = (0.4, 0.4, 0.2)

steps = np.arange(0.0, 1.01, 0.1)
for w_lgb in steps:
    for w_xgb in steps:
        w_cb = round(1.0 - w_lgb - w_xgb, 10)
        if w_cb < -1e-9 or w_cb > 1.0 + 1e-9:
            continue
        w_cb = max(0.0, min(1.0, w_cb))
        y_blend = w_lgb * val_lgb + w_xgb * val_xgb + w_cb * val_cb
        val_r2 = r2_score(y_val.values, y_blend)
        if val_r2 > best_val_r2:
            best_val_r2 = val_r2
            best_w = (float(w_lgb), float(w_xgb), float(w_cb))

w_lgb_opt, w_xgb_opt, w_cb_opt = best_w
print(f'Optimal weights (val R²={best_val_r2:.4f}):  LGB={w_lgb_opt:.2f}  XGB={w_xgb_opt:.2f}  CB={w_cb_opt:.2f}')

# Test set evaluation
y_pred_ens3 = (
    w_lgb_opt * y_pred_lgb_opt +
    w_xgb_opt * y_pred_xgb_opt +
    w_cb_opt  * y_pred_cb
)
res_ens3 = evaluate('Ensemble (Optuna LGB+XGB+CatBoost)', y_test.values, y_pred_ens3)
results.append(res_ens3)
print(f"Test R²: {res_ens3['r2']:.4f}  MAE: ${res_ens3['mae_dollar']:,.0f}  Pred mean: ${res_ens3['pred_mean']:,.0f}")

Optimal weights (val R²=0.5591):  LGB=1.00  XGB=0.00  CB=0.00
Test R²: 0.4443  MAE: $6,163  Pred mean: $2,745


## 10. Stacking Ensemble (5-fold OOF + Ridge meta-learner)

Stacking generates out-of-fold (OOF) predictions from 4 base models via 5-fold CV, then trains a Ridge meta-learner on those OOF predictions. This exploits model complementarity more flexibly than fixed weights.

`positive=True` in Ridge prevents the meta-learner from shorting models.

In [32]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

X_arr      = X_train_adv.values
y_arr      = y_train.values
X_test_arr = X_test_adv.values

# Build param dicts with plain Python scalars (no numpy types)
_lgb_params  = {k: v for k, v in lgb_best_params.items()}
_xgb_params  = {k: v for k, v in xgb_best_params.items() if k != 'early_stopping_rounds'}

base_configs = {
    'LGB_opt': lgb.LGBMRegressor(**_lgb_params, verbose=-1, n_jobs=-1, random_state=42),
    'XGB_opt': XGBRegressor(**_xgb_params, n_jobs=-1, random_state=42, verbosity=0),
    'CatBoost': cb.CatBoostRegressor(
        iterations=1000, learning_rate=0.03, depth=7,
        l2_leaf_reg=3.0, random_seed=42, verbose=0
    ),
    'RF': RandomForestRegressor(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=42),
}

oof_preds       = {name: np.zeros(len(y_arr)) for name in base_configs}
test_preds_list = {name: [] for name in base_configs}

for fold_idx, (tr_idx, val_idx) in enumerate(kf.split(X_arr)):
    X_fold_tr, X_fold_val = X_arr[tr_idx], X_arr[val_idx]
    y_fold_tr = y_arr[tr_idx]
    print(f'Fold {fold_idx + 1}/5 ...', end=' ', flush=True)

    for name, model_proto in base_configs.items():
        model = copy.deepcopy(model_proto)
        model.fit(X_fold_tr, y_fold_tr)
        oof_preds[name][val_idx] = model.predict(X_fold_val)
        test_preds_list[name].append(model.predict(X_test_arr))
    print('done')

print()
print('OOF R² per base model:')
for name in base_configs:
    oof_r2 = r2_score(y_arr, oof_preds[name])
    print(f'  {name:<12} OOF R²: {oof_r2:.4f}')

# ── Meta-learning ────────────────────────────────────────────────────────────
oof_stack  = np.column_stack([oof_preds[n] for n in base_configs])
test_stack = np.column_stack([np.mean(test_preds_list[n], axis=0) for n in base_configs])

meta = Ridge(alpha=1.0, positive=True)
meta.fit(oof_stack, y_arr)
print('\nMeta weights:', dict(zip(base_configs.keys(), meta.coef_.round(4))))

y_pred_stack = meta.predict(test_stack)
res_stack = evaluate('Stacking (4 models + Ridge meta)', y_test.values, y_pred_stack)
results.append(res_stack)
print(f"Test R²: {res_stack['r2']:.4f}  MAE: ${res_stack['mae_dollar']:,.0f}  Pred mean: ${res_stack['pred_mean']:,.0f}")

Fold 1/5 ... done
Fold 2/5 ... done
Fold 3/5 ... done
Fold 4/5 ... done
Fold 5/5 ... done

OOF R² per base model:
  LGB_opt      OOF R²: 0.4232
  XGB_opt      OOF R²: 0.4239
  CatBoost     OOF R²: 0.4271
  RF           OOF R²: 0.4061

Meta weights: {'LGB_opt': np.float64(0.1947), 'XGB_opt': np.float64(0.1011), 'CatBoost': np.float64(0.5986), 'RF': np.float64(0.1131)}
Test R²: 0.4497  MAE: $6,130  Pred mean: $2,865


## 11. Optional — MEPS Raw Feature Enrichment

If the raw MEPS file (`h243.dta`) is available, we can add features that are not currently in the engineered dataset but are known cost drivers:

1. **Smoking status** (`ADSMOK42`) — active smokers have ~30–50% higher annual spending; NOT in current feature set
2. **Physical activity** (`ADEXRC42`) — protective factor; sedentary individuals trend toward higher utilisation
3. **More granular functional limitations** (`WLKLIM22`, `LFTDIF22`, `ACTLIM22`) — current features only capture ADL/IADL help needed
4. **Prescription fill count** (`RXTOT22`) — strongest proxy for disease burden beyond binary diagnosis flags; chronic poly-pharmacy is a major cost signal

These features could add +0.02–0.05 R² based on prior MEPS research.

In [33]:
import os

dta_candidates = [
    os.path.join('..', 'notebooks', 'h243.dta'),
    'h243.dta',
    os.path.join('..', 'data', 'raw', 'h243.dta'),
]
dta_path = next((p for p in dta_candidates if os.path.exists(p)), None)

if dta_path is None:
    print('WARNING: h243.dta not found — skipping raw feature enrichment.')
    print('Place h243.dta in notebooks/ or data/raw/ and re-run this section.')
    enriched_available = False
else:
    print(f'Loading raw MEPS file from: {dta_path}')
    raw = pd.read_stata(dta_path, convert_categoricals=False)

    raw_cols = ['DUPERSID', 'ADSMOK42', 'ADEXRC42',
                'WLKLIM22', 'LFTDIF22', 'ACTLIM22',
                'RXTOT22', 'OBTOTV22']

    available_cols = [c for c in raw_cols if c in raw.columns]
    missing_cols   = [c for c in raw_cols if c not in raw.columns]
    print(f'Available raw cols: {available_cols}')
    if missing_cols:
        print(f'Missing cols (skipped): {missing_cols}')

    raw_sub = raw[available_cols].copy()
    raw_sub.columns = [c.lower() for c in raw_sub.columns]

    # Map ADSMOK42: 1 = current smoker, others = 0/-1 flagged as 0
    if 'adsmok42' in raw_sub.columns:
        raw_sub['current_smoker'] = (raw_sub['adsmok42'] == 1).astype(float)

    # Map ADEXRC42: 1 = exercises regularly, others = 0
    if 'adexrc42' in raw_sub.columns:
        raw_sub['exercises_regularly'] = (raw_sub['adexrc42'] == 1).astype(float)

    # Functional limitation variables: positive integer = limited, recode to 0/1
    for col in ['wlklim22', 'lftdif22', 'actlim22']:
        if col in raw_sub.columns:
            raw_sub[col] = (raw_sub[col] > 0).astype(float)

    # RXTOT22: total prescription fills — use as continuous, clip negative/invalid
    if 'rxtot22' in raw_sub.columns:
        raw_sub['rx_fills'] = raw_sub['rxtot22'].clip(lower=0).fillna(0).astype(float)

    print(raw_sub.head())
    enriched_available = True


if enriched_available:
    # Re-run Optuna LGB with enriched features
    # (Requires merging DUPERSID from original MEPS into the train/test splits —
    #  this is left as an exercise if the raw file is present.)
    print('\nEnriched feature merge requires DUPERSID linkage to X_train/X_test.')
    print('Re-run 03_feature_engineering.ipynb with these columns to get a complete enriched dataset.')

Loading raw MEPS file from: ../notebooks/h243.dta
Available raw cols: ['DUPERSID', 'RXTOT22', 'OBTOTV22']
Missing cols (skipped): ['ADSMOK42', 'ADEXRC42', 'WLKLIM22', 'LFTDIF22', 'ACTLIM22']
     dupersid  rxtot22  obtotv22  rx_fills
0  2460002101       40         2      40.0
1  2460006101       43        28      43.0
2  2460006102        1         1       1.0
3  2460010101       36        16      36.0
4  2460018101       23         8      23.0

Enriched feature merge requires DUPERSID linkage to X_train/X_test.
Re-run 03_feature_engineering.ipynb with these columns to get a complete enriched dataset.


## 12. Full Comparison + Analysis

In [34]:
# Load historical results from model_results.json
results_path = os.path.join('..', 'models', 'model_results.json')
with open(results_path) as f:
    historical = json.load(f)

new_model_names = {r['model'] for r in results}
all_results = [r for r in historical if r['model'] not in new_model_names] + results

# Sort by R² descending
all_results_sorted = sorted(all_results, key=lambda x: x['r2'], reverse=True)

print('=' * 90)
print('FULL MODEL COMPARISON — ALL NOTEBOOKS'.center(90))
print('=' * 90)
print(f'  {"Model":<42} {"R²":>8} {"RMSE_log":>10} {"MAE ($)":>10} {"Pred Mean":>11}')
print('-' * 90)
for r in all_results_sorted:
    pred_mean = r.get('pred_mean', float('nan'))
    pred_str  = f'${pred_mean:>9,.0f}' if pred_mean == pred_mean else '        N/A'
    print(f"  {r['model']:<42} {r['r2']:>8.4f} {r['rmse_log']:>10.4f}"
          f" ${r['mae_dollar']:>9,.0f} {pred_str}")
print('=' * 90)
print(f'  Actual test mean spending: ${actual_mean:,.0f}')

best_new = max(results, key=lambda x: x['r2'])
baseline_r2 = 0.4049  # Linear Regression from notebook 04

print(f'\nBest model overall : {all_results_sorted[0]["model"]}  (R²={all_results_sorted[0]["r2"]:.4f})')
print(f'Best new model     : {best_new["model"]}  (R²={best_new["r2"]:.4f})')
print(f'\nR² progression:')
print(f'  Baseline (Linear Regression)  : {baseline_r2:.4f}')
print(f'  Notebook 05 best (LGB+XGB ens): 0.4440')
print(f'  This notebook best            : {best_new["r2"]:.4f}')
print(f'  Total gain from baseline      : +{best_new["r2"] - baseline_r2:.4f}')

                          FULL MODEL COMPARISON — ALL NOTEBOOKS                           
  Model                                            R²   RMSE_log    MAE ($)   Pred Mean
------------------------------------------------------------------------------------------
  Stacking (4 models + Ridge meta)             0.4497     2.3831 $    6,130 $    2,865
  CatBoost (85 feat)                           0.4460     2.3911 $    6,163 $    2,921
  XGB Optuna (85 feat)                         0.4452     2.3929 $    6,140 $    2,968
  LGB Optuna (85 feat)                         0.4443     2.3949 $    6,163 $    2,745
  Ensemble (Optuna LGB+XGB+CatBoost)           0.4443     2.3949 $    6,163 $    2,745
  XGBoost                                      0.4398     2.4044 $    6,220         N/A
  Random Forest                                0.4191     2.4485 $    6,286         N/A
  Linear Regression                            0.4049     2.4783 $    6,980         N/A
  SVR                          

## 13. Save Best Models + Update Artifacts

In [35]:
models_dir = os.path.join('..', 'models')
os.makedirs(models_dir, exist_ok=True)

feature_cols_adv = list(X_train_adv.columns)   # 85 features

# ── Save model pkl files ─────────────────────────────────────────────────────
joblib.dump(lgb_opt,  os.path.join(models_dir, 'lightgbm_opt.pkl'))
joblib.dump(xgb_opt,  os.path.join(models_dir, 'xgboost_opt.pkl'))
joblib.dump(cb_model, os.path.join(models_dir, 'catboost.pkl'))
joblib.dump(meta,     os.path.join(models_dir, 'stacking_meta.pkl'))
print('Saved: lightgbm_opt.pkl, xgboost_opt.pkl, catboost.pkl, stacking_meta.pkl')

# ── Update model_meta.json ────────────────────────────────────────────────────
meta_path = os.path.join(models_dir, 'model_meta.json')
with open(meta_path) as f:
    meta_json = json.load(f)

meta_json['feature_cols_advanced'] = feature_cols_adv

meta_json['models']['LightGBM Opt'] = {
    'file': 'lightgbm_opt.pkl',
    'needs_scaling': False,
    'target_space': 'log1p',
    'smearing_factor': None,
    'feature_set': 'advanced',
    'best_params': lgb_best_params,
}
meta_json['models']['XGBoost Opt'] = {
    'file': 'xgboost_opt.pkl',
    'needs_scaling': False,
    'target_space': 'log1p',
    'smearing_factor': None,
    'feature_set': 'advanced',
    'best_params': xgb_best_params,
}
meta_json['models']['CatBoost'] = {
    'file': 'catboost.pkl',
    'needs_scaling': False,
    'target_space': 'log1p',
    'smearing_factor': None,
    'feature_set': 'advanced',
}
meta_json['models']['Stacking'] = {
    'meta_file': 'stacking_meta.pkl',
    'base_models': list(base_configs.keys()),
    'feature_set': 'advanced',
    'note': '5-fold OOF stacking with Ridge meta-learner (positive=True)',
    'meta_weights': {name: float(coef) for name, coef in zip(base_configs.keys(), meta.coef_)},
}
meta_json['ensemble_advanced'] = {
    'models': ['LightGBM Opt', 'XGBoost Opt', 'CatBoost'],
    'weights': [w_lgb_opt, w_xgb_opt, w_cb_opt],
    'feature_set': 'advanced',
    'note': 'Weights grid-searched on validation split (15% of train)',
}

with open(meta_path, 'w') as f:
    json.dump(meta_json, f, indent=2)
print('Updated: model_meta.json')

# ── Update model_results.json ─────────────────────────────────────────────────
with open(results_path) as f:
    orig_results = json.load(f)

new_names = {r['model'] for r in results}
orig_results = [r for r in orig_results if r['model'] not in new_names]
orig_results.extend(results)

with open(results_path, 'w') as f:
    json.dump(orig_results, f, indent=2)
print('Updated: model_results.json')

# ── Summary ───────────────────────────────────────────────────────────────────
print()
print('Files saved / updated in models/:')
print('  lightgbm_opt.pkl      — Optuna-tuned LightGBM (85 features)')
print('  xgboost_opt.pkl       — Optuna-tuned XGBoost  (85 features)')
print('  catboost.pkl          — CatBoost baseline     (85 features)')
print('  stacking_meta.pkl     — Ridge meta-learner for stacking')
print('  model_meta.json       — updated: feature_cols_advanced, new model entries')
print('  model_results.json    — updated: all new model results appended')
print()
print(f'Best model in this notebook: {best_new["model"]}  (R²={best_new["r2"]:.4f})')
print(f'Improvement over notebook 05 ensemble: +{best_new["r2"] - 0.444:.4f}')

Saved: lightgbm_opt.pkl, xgboost_opt.pkl, catboost.pkl, stacking_meta.pkl
Updated: model_meta.json
Updated: model_results.json

Files saved / updated in models/:
  lightgbm_opt.pkl      — Optuna-tuned LightGBM (85 features)
  xgboost_opt.pkl       — Optuna-tuned XGBoost  (85 features)
  catboost.pkl          — CatBoost baseline     (85 features)
  stacking_meta.pkl     — Ridge meta-learner for stacking
  model_meta.json       — updated: feature_cols_advanced, new model entries
  model_results.json    — updated: all new model results appended

Best model in this notebook: Stacking (4 models + Ridge meta)  (R²=0.4497)
Improvement over notebook 05 ensemble: +0.0057
